# Musical Profile Clustering

## CMOR 438 Final Project: Music Analytics with a From-Scratch Machine Learning Package

This notebook clusters songs into interpretable musical profiles using the custom `KMeans` implementation from `music_ml`.

## 1) Motivation

Clustering can reveal naturally occurring groups of songs based on audio characteristics. These groups can then be interpreted as musical profiles (e.g., energetic dance tracks, acoustic mellow tracks, etc.).

## 2) Imports and Dataset Loading

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans as SklearnKMeans
from sklearn.decomposition import PCA

# Make src package importable from notebooks/
sys.path.insert(0, str(Path("..").resolve() / "src"))

from music_ml.preprocessing import StandardScaler
from music_ml.unsupervised import KMeans

data_path = Path("../data/processed/spotify_processed.csv")

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"Loaded dataset from: {data_path}")
    print(f"Shape: {df.shape}")
else:
    df = pd.DataFrame()
    print(
        "Dataset file not found. Place your processed CSV in data/processed/ "
        "and update data_path if needed."
    )

## 3) Select Clustering Features

Use numeric audio descriptors that capture timbre, rhythm, and perceived mood.

In [ ]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
]

if df.empty:
    X = np.array([])
    available_features = []
    print("No dataset loaded yet.")
else:
    available_features = [col for col in audio_features if col in df.columns]
    if len(available_features) == 0:
        X = np.array([])
        print("No expected audio feature columns found.")
    else:
        clustering_df = df[available_features].dropna().copy()
        X = clustering_df.to_numpy(dtype=float)
        print(f"Using {len(available_features)} features.")
        print(f"Samples available for clustering: {X.shape[0]}")
        print("Features:", available_features)

## 4) Standardize Features

KMeans is distance-based, so feature scaling is important.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    print("Feature scaling complete.")
    print("Scaled means (approx. 0):", np.round(X_scaled.mean(axis=0), 3))

## 5) Candidate Values of k (Elbow-Style Analysis)

We inspect inertia across multiple values of `k` to guide model selection.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    k_values = list(range(2, 11))
    inertias = []

    for k in k_values:
        model_k = KMeans(n_clusters=k, max_iters=200, tol=1e-4, random_state=42)
        model_k.fit(X_scaled)
        inertias.append(model_k.inertia_)

    plt.figure(figsize=(7, 4))
    plt.plot(k_values, inertias, marker="o")
    plt.xlabel("Number of clusters (k)")
    plt.ylabel("Inertia")
    plt.title("Elbow-Style Plot (Custom KMeans)")
    plt.xticks(k_values)
    plt.grid(alpha=0.3)
    plt.show()

    elbow_table = pd.DataFrame({"k": k_values, "inertia": inertias})
    display(elbow_table)

## 6) Fit Custom KMeans

Choose `k` based on the elbow trend and project goals.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    chosen_k = 4
    custom_kmeans = KMeans(n_clusters=chosen_k, max_iters=300, tol=1e-4, random_state=42)
    custom_kmeans.fit(X_scaled)

    custom_labels = custom_kmeans.labels_

    print(f"Chosen k: {chosen_k}")
    print(f"Iterations: {custom_kmeans.n_iter_}")
    print(f"Inertia: {custom_kmeans.inertia_:.4f}")

## 7) Inspect Cluster Centroids

Centroids are shown in standardized space and transformed back to original feature units for interpretation.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    centroids_scaled = pd.DataFrame(
        custom_kmeans.centroids_,
        columns=available_features,
        index=[f"cluster_{i}" for i in range(chosen_k)],
    )

    centroids_original = pd.DataFrame(
        custom_kmeans.centroids_ * scaler.scale_ + scaler.mean_,
        columns=available_features,
        index=[f"cluster_{i}" for i in range(chosen_k)],
    )

    print("Centroids (standardized feature space):")
    display(centroids_scaled)

    print("Centroids (original feature units):")
    display(centroids_original)

## 8) Visualize Clusters in 2D (PCA for Visualization)

PCA is used only to project high-dimensional features into 2D for visualization.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X_scaled)

    plt.figure(figsize=(7, 5))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=custom_labels, alpha=0.7, cmap="tab10")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Custom KMeans Clusters in PCA Space")
    plt.colorbar(scatter, label="Cluster")
    plt.show()

    explained = pca.explained_variance_ratio_.sum()
    print(f"Total variance explained by 2 PCs: {explained:.3f}")

## 9) Compare with scikit-learn KMeans

We compare inertia and broad cluster structure as a sanity check against a standard implementation.

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    sk_kmeans = SklearnKMeans(n_clusters=chosen_k, n_init=10, random_state=42)
    sk_labels = sk_kmeans.fit_predict(X_scaled)

    comparison = pd.DataFrame(
        {
            "model": ["custom_kmeans", "sklearn_kmeans"],
            "inertia": [custom_kmeans.inertia_, float(sk_kmeans.inertia_)],
        }
    )
    display(comparison)

    plt.figure(figsize=(7, 5))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=sk_labels, alpha=0.7, cmap="tab10")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("scikit-learn KMeans Clusters in PCA Space")
    plt.colorbar(scatter, label="Cluster")
    plt.show()

## 10) Interpret Clusters as Musical Profiles

Use centroid patterns to define practical profile names and descriptions.

Suggested interpretation workflow:

- identify high/low features per cluster in `centroids_original`
- assign profile labels (e.g., "high-energy dance", "acoustic mellow")
- validate interpretability by checking representative songs from each cluster
- note limitations: sensitivity to feature choice, scaling, and selected `k`

In [ ]:
if X.size == 0:
    print("Clustering data is not ready.")
else:
    profile_summary = centroids_original.copy()
    profile_summary["profile_name"] = "(fill in)"
    profile_summary["interpretation_notes"] = "(fill in)"

    display(profile_summary)

    print("Next step: rename each cluster with a musical profile based on centroid characteristics.")